# A2: Unsupervised Learning on Time Series Data — Anomaly Detection

**Course:** Machine Learning for Engineers

**Dataset:** NASA Anomaly Detection Dataset — SMAP & MSL  
**Source:** [Kaggle — patrickfleith/nasa-anomaly-detection-dataset-smap-msl](https://www.kaggle.com/datasets/patrickfleith/nasa-anomaly-detection-dataset-smap-msl)  


---
# 1. Frame the Problem and Look at the Big Picture

## 1.1 Business Objective

**Goal:** Detect anomalies in telemetry time series collected from two NASA spacecraft:
- **SMAP** (Soil Moisture Active Passive) — an Earth-orbiting satellite
- **MSL** (Mars Science Laboratory) — the Curiosity rover on Mars

In operational practice, anomaly detection on spacecraft telemetry enables ground engineers to identify unexpected system behaviour early, investigate potential faults, and prevent mission-critical failures without requiring continuous human monitoring of hundreds of channels.

**Dataset summary:**

| Spacecraft | Channels | Total timesteps | Labeled anomalies |
|------------|----------|-----------------|-------------------|
| SMAP       | 55       | 429,735         | 69                |
| MSL        | 27       | 66,709          | 36                |


## 1.2 How the Solution Will Be Used

In this assignment, models are evaluated offline: trained on the clean training split and scored against the test split using ground-truth labels from `labeled_anomalies.csv`.

In a real operations context, the system would monitor live telemetry and flag unusual channels automatically, reducing the need for manual inspection.

## 1.3 Current Solutions / Baselines

NASA's existing monitoring relies on fixed thresholds and manual review of ISA reports. This works for obvious point anomalies but not for contextual ones. Hundman et al. (2018) addressed this with an LSTM and dynamic thresholding approach; this assignment compares simpler unsupervised alternatives.

## 1.4 Problem Type

This is an unsupervised anomaly detection problem on multivariate time series. Models train on clean telemetry only and flag deviations at test time; no labels are used during training.

Two types of anomalies appear in the dataset:
- **Point anomalies**: individual timesteps that are statistically extreme
- **Contextual anomalies**: values that look normal in isolation but are anomalous given the surrounding context

Techniques to implement and compare:
1. **K-Means** — distance to nearest centroid as anomaly score 
2. **DBSCAN** — noise points (label = -1) treated as anomalies 
3. **Gaussian Mixture Model (GMM)** — negative log-likelihood as anomaly score 
4. **Isolation Forest** — anomaly score from random path lengths 
5. **Local Outlier Factor (LOF)** — local density deviation as anomaly score 
6. **SVM** - 
7. **PCA** - 
8. **STOMPY** - 

## 1.5 Performance Measurement

Models are compared using:

- **F1 score** (primary metric): balances precision and recall, appropriate for the imbalanced anomaly detection setting
- **Precision and Recall**: in spacecraft operations, recall is prioritised since missing an anomaly is more costly than a false alert

Ground-truth labels from `labeled_anomalies.csv` are used for evaluation only, not during training.

## 1.6 Minimum Performance Requirements

Since ground-truth labels are available in `labeled_anomalies.csv`, we can set concrete targets for the models:

- **F1 score > 0.5** on the test set - a random classifier at ~5% contamination would score ~0.1
- **Recall > 0.6** - missing anomalies is costly in spacecraft operations; recall is prioritised over precision
- **False alarm rate < 10%** - too many false alerts would make the system impractical for operators
- Models should detect anomalies **without being told where they are** (trained only on the clean training split)

---
# 2. Libraries

In [ ]:
# Main libraries
import numpy as np
import pandas as pd

import sys
import os
# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

from Dataset import DatasetOperations
from PCA_method import BatchPCA
from Evaluation import EvaluateResults
from GausianMixtureModel import GMM
from LocalOutlierFactor import LOF
from TimeContext import TimeContextModif
import numpy as np
from Stompy import MstumpDetector
from LSTMAutoencoder import LSTM_AE_Detector

import warnings
warnings.filterwarnings('ignore')


---
# 3. Get the Data

## 3.1 Load Dataset

The dataset contains 82 channels across two spacecraft (55 SMAP, 27 MSL). Each channel is stored as a separate `.npy` file containing the raw telemetry readings alongside pre-engineered command and context features.

Labels are provided in `labeled_anomalies.csv` as index ranges into the test array for each channel. These are used for evaluation only.

In [ ]:


# Make sure Dataset.py is importable from the project directory
BASE_DIR = os.getcwd()
if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)



# Paths (mirrors Main.py)
train_path       = os.path.join(BASE_DIR, "archive", "data", "data", "train")
test_path        = os.path.join(BASE_DIR, "archive", "data", "data", "test")
result_file_path = os.path.join(BASE_DIR, "archive", "labeled_anomalies.csv")

dt = DatasetOperations(test_path=test_path, train_path=train_path, results_path=result_file_path)
train_dataset, test_dataset, results = dt.load_data()

evaluation = EvaluateResults()
evaluation.load_solution(results)

## 3.2 Dataset Info

In [ ]:
dt.dataset_info()

No missing values or non-binary command features were found across any of the 82 channels. Telemetry values are largely within [-1, 1], indicating the data has been pre-normalized. A handful of channels show a constant telemetry range (e.g. [-1.00, -1.00]), meaning the raw signal does not vary at all in the training split.

Both point and contextual anomalies are present in the labels.

---
# 4. Explore the Data (EDA)

## 4.1 Time Series Visualization

Plot the raw telemetry signal (column 0) for the selected channel(s) over time.  
Overlay the known anomaly windows from `labeled_anomalies.csv` as shaded regions on the **test** portion — this gives a visual intuition of what we are trying to detect.

In [ ]:
dt.plot_data_signal(choosen_dataset="training_dataset", channel_id="P-1")

## 4.2 Correlation Matrix 
corelation check function creates correlation matrix and creates a csv report about correlation between files

In [ ]:
correlation_outfile_path = BASE_DIR
correlation_dictionary = dt.correlation_check(mode="training_dataset",correlation_csv_report = False,correlation_outfile_path=correlation_outfile_path,corr_calc_method="spearman")

## 4.3 Removing constant columns and lowering amount of data 
code removes columns that have same constant value throughout whole dataset in training and testing dataset simultaneously.

In [ ]:
train_subset_cl,test_subset_cl = dt.remove_constant_columns(train_dataset,test_dict=test_dataset)

---
# 5. Prepare the Data

## 5.1 Train / Test Split 

The NASA dataset **already provides a pre-defined chronological train/test split** — no splitting needed:

- `archive/data/data/train/{chan_id}.npy` → training data (anomaly-free, used for fitting models)
- `archive/data/data/test/{chan_id}.npy` → test data (may contain anomalies, evaluated against labels)

Ground-truth labels for the **test set only** are in `labeled_anomalies.csv`.  
The training set is considered clean — this is the standard assumption for unsupervised anomaly detection (train on normal, detect deviations at test time).
First round of method usage will be perform on smaller dataset made up of 10 randomly selected files
Also to lower amount of data function sort_by_corr creates a groups of files with high correlation

In [ ]:
import ast

CHANNEL_ID = 'D-9'

# Ground-truth label vector for single-channel plots
row    = results[results['chan_id'] == CHANNEL_ID]
y_test = np.zeros(len(test_dataset[CHANNEL_ID]), dtype=int)
if not row.empty:
    for start, end in ast.literal_eval(row.iloc[0]['anomaly_sequences']):
        y_test[start:end + 1] = 1

print(f"Channel      : {CHANNEL_ID}")
print(f"Train shape  : {train_dataset[CHANNEL_ID].shape}")
print(f"Test shape   : {test_dataset[CHANNEL_ID].shape}")
print(f"Anomaly class: {row.iloc[0]['class']}")
print(f"Anomalies    : {y_test.sum()} / {len(y_test)}  ({100*y_test.mean():.1f}%)")

# 10-channel subset for model training and evaluation (same seed as Tomas)
train_subset, test_subset = train_dataset, test_dataset
print(f"\nSubset channels: {list(train_subset.keys())}")

In [ ]:
train_data_clustered_by_corr = dt.sort_by_corr(sorting_threshold=0.60, remove_files=False,remove_threshold=None)
train_subset_cl,test_subset_cl = dt.select_subset(random_selection=True, manual_file_names=None, subset_size=20, seed=42)


## 5.2 Feature Engineering

**Motion and Change**: Adds velocity and acceleration (derivatives) to capture how fast or abruptly sensor values are changing.

**Vibrations and Energy**: Uses frequency analysis (FFT) to reveal unusual patterns in signal "jitter" or rhythmic noise.

**Trends and Stability**: Adds rolling statistics (mean, variance) to help the model spot when a signal starts to drift or becomes unstable.

In [ ]:
tx = TimeContextModif(test_dataset=test_subset_cl,train_dataset=train_subset_cl)
train_s_der, test_s_der = tx.add_derivative_features()

tcx = TimeContextModif(test_dataset=test_s_der,train_dataset=train_s_der)
train_s, test_s = tcx.add_spectral_features(window_length=250)

# tcx4 = TimeContextModif(test_dataset=test_s_sw,train_dataset=train_s_sw)
# train_s, test_s = tcx4.add_rolling_statistics(window_length=200)


##  Dataset Reduction
 **PCA-based triage** performs by identifying channels where the model successfully detects at least one true anomaly and then creates a filtered subset containing only these "surviving" datasets for subsequent, more intensive analysis.

In [ ]:
bpca_sort = BatchPCA(train_s, test_s)
bpca_sort.fit_all(n_components=30)
PCA_sort_test_errors = bpca_sort.get_PCA_predictions(mode="test",threshold_percentile=80)
PCA_sort_result = evaluation.compare_methods_results(predictions_dict=PCA_sort_test_errors)

surviving_channels = PCA_sort_result[PCA_sort_result['TP'] > 0]['Channel'].tolist()
train_surv = {cid: train_s[cid] for cid in surviving_channels}
test_surv = {cid: test_s[cid] for cid in surviving_channels}

---
# 6. Explore Many Different Models and Shortlist the Best Ones

## 6.1 PCA

In [ ]:
bpca = BatchPCA(train_surv, test_surv)
bpca.fit_all(n_components=15)
PCA_test_errors = bpca.get_PCA_predictions(mode="test",threshold_percentile=95)
PCA_result = evaluation.compare_methods_results(predictions_dict=PCA_test_errors)
evaluation.plot_hits_vs_misses(PCA_result)
print(PCA_result)

The high Micro F1 (0.75) contrasted with a low Macro F1 (0.55) proves that detection accuracy is inconsistent across the 80 channels. While PCA captures over 80% variance for most signals, several channels suffer from excessive information loss below 60%, undermining their reconstruction reliability. Consequently, the current configuration fails to achieve uniform performance across the diverse dataset.

## 6.2 K-Means Clustering

In [ ]:
from KMeans import KMeansAnalyzer

# Elbow plot on D-9 to choose K
km_analyzer = KMeansAnalyzer(train_surv, test_surv)
km_analyzer.elbow_plot(CHANNEL_ID, k_range=range(2, 12))

In [ ]:
from sklearn.decomposition import PCA

K_BEST = 5

km_analyzer.fit_all(k=K_BEST)

# Compute 2D PCA projection of training data for cluster visualisation
mask = km_analyzer.feature_masks[CHANNEL_ID]
scaler = km_analyzer.scalers[CHANNEL_ID]
X_scaled = scaler.transform(train_surv[CHANNEL_ID][:, mask])
X_train_2d = PCA(n_components=2).fit_transform(X_scaled)

km_analyzer.plot_clusters(CHANNEL_ID, X_train_2d)

train_km, test_km = km_analyzer.get_enriched_features()

km_preds  = km_analyzer.get_batch_predictions(threshold_percentile=95)
km_report = evaluation.compare_methods_results(predictions_dict=km_preds)
evaluation.plot_hits_vs_misses(km_report, title='K-Means — Centroid Distance')
print(km_report[['Channel', 'Precision', 'Recall', 'F1_Score']].to_string(index=False))

**K-Means Analysis**

Recall is 1.0 everywhere: the model flagged so many points that it hit every anomaly window by accident. Precision is the real problem: G-1 at 1.4% means 70 false alarms per true anomaly.

D-9 and D-15 perform better (precision 0.78, 0.60) because their anomalies sit far from the normal clusters. G-1, T-5, and T-3 fail because the anomalous points overlap with the normal operating modes.

## 6.3 DBSCAN

In [ ]:
from DBSCANModel import DBSCANModel

# K-distance plot: look for the 'knee' — that y-value is the natural eps for this channel
db_analyzer = DBSCANModel(train_surv, test_surv)
db_analyzer.k_distance_plot(CHANNEL_ID, k=5)

In [ ]:
EPS         = 2.0
MIN_SAMPLES = 10

db_analyzer = DBSCANModel(train_surv, test_surv)
db_analyzer.fit_all(eps=EPS, min_samples=MIN_SAMPLES)
db_preds   = db_analyzer.get_batch_predictions()
db_report  = evaluation.compare_methods_results(predictions_dict=db_preds)
evaluation.plot_hits_vs_misses(db_report, title='DBSCAN')
print(db_report[['Channel', 'Precision', 'Recall', 'F1_Score']].to_string(index=False))

**DBSCAN Analysis**

Macro F1 = 0.69, a clear improvement over K-Means. D-9 (F1=0.999), D-3 (0.974), and D-15 (0.887) work well — anomalies in these channels form isolated points outside the main clusters.

G-7 and T-5 both score F1=0.0. G-7 produced 1 cluster and 0 noise points — eps=2.0 is too large so everything is absorbed. T-5 flagged 41 noise points but none matched the real anomalies. Using a single eps across all channels is the core limitation here.

## 6.4 Gaussian Mixture Model (GMM)

In [ ]:
#optimization of parameters
# def objective_gmm(trial):
#     n_components = trial.suggest_int('n_components', 2, 6)
#     covariance_type = trial.suggest_categorical('covariance_type', ['full', 'tied', 'diag'])
    
#     try:
#         gmm = GMM(train_pca_features, test_pca_features)
#         gmm.fit_all(
#             n_components=n_components,
#             covariance_type=covariance_type,
#             n_init=3,
#             max_iter=300,
#             tol=1e-3
#         )
#         preds = gmm.get_batch_predictions(threshold_percentile=5)
#         report = evaluation.compare_methods_results(preds)
        
#         if report is None or report.empty:
#             return 0.0
#         return report['F1_Score'].mean()
        
#     except Exception:
#         return 0.0

# study_gmm = optuna.create_study(direction='maximize')
# study_gmm.optimize(objective_gmm, n_trials=30)
# print(study_gmm.best_params)

# Final model
gmm = GMM(train_surv,test_surv)
gmm.fit_all(n_components=3, covariance_type='diag', n_init=5, max_iter=300, tol=1e-3, init_params='kmeans', warm_start=True)
GMM_errors_pred = gmm.get_batch_predictions(threshold_percentile=5)
GMM_results = evaluation.compare_methods_results(predictions_dict=GMM_errors_pred)
evaluation.plot_hits_vs_misses(GMM_results)
print(GMM_results)


The high Micro F1 (0.75) contrasted with a low Macro F1 (0.55) proves that detection accuracy is inconsistent across the 80 channels. While PCA captures over 80% variance for most signals, several channels suffer from excessive information loss below 60%, undermining their reconstruction reliability. Consequently, the current configuration fails to achieve uniform performance across the diverse dataset.

## 6.5 Isolation Forest

In [ ]:
from IsolationForestModel import IsolationForestModel

CONTAMINATION = 0.10

iso = IsolationForestModel(train_km, test_km)
iso.fit_all(contamination=CONTAMINATION)
iso_preds   = iso.get_batch_predictions()
iso_report  = evaluation.compare_methods_results(predictions_dict=iso_preds)
evaluation.plot_hits_vs_misses(iso_report, title='Isolation Forest')
print(iso_report[['Channel', 'Precision', 'Recall', 'F1_Score']].to_string(index=False))

**Isolation Forest Analysis**

Macro F1 = 0.43. Same recall=1.0 pattern as K-Means — everything gets flagged, which catches every anomaly but floods the results with false positives.

D-3 (F1=0.92) and D-9 (0.84) hold up due to large anomaly regions that keep precision reasonable despite the false alarms. G-1 (precision 0.05) and E-11 (0.08) are the worst — almost all flagged points are normal data.

## 6.6 Model 5: Local Outlier Factor (LOF)

In [ ]:
lof = LOF(train_surv,test_surv)
lof.fit_all(n_neighbors=50, algorithm='auto', leaf_size=50, metric='minkowski', p=4, contamination='auto')
LOF_error_prediction = lof.get_batch_predictions(threshold_percentile=10)
LOF_results = evaluation.compare_methods_results(predictions_dict=LOF_error_prediction)
evaluation.plot_hits_vs_misses(LOF_results)
print(LOF_results)

The high Micro F1 (0.75) contrasted with a low Macro F1 (0.55) proves that detection accuracy is inconsistent across the 80 channels. While PCA captures over 80% variance for most signals, several channels suffer from excessive information loss below 60%, undermining their reconstruction reliability. Consequently, the current configuration fails to achieve uniform performance across the diverse dataset.

## 6.7 Model 6: One-Class SVM

In [ ]:
from OneClassSVMModel import OneClassSVMModel

NU = 0.05

svm = OneClassSVMModel(train_km, test_km)
svm.fit_all(nu=NU)
svm_preds   = svm.get_batch_predictions()
svm_report  = evaluation.compare_methods_results(predictions_dict=svm_preds)
evaluation.plot_hits_vs_misses(svm_report, title='One-Class SVM')
print(svm_report[['Channel', 'Precision', 'Recall', 'F1_Score']].to_string(index=False))

**One-Class SVM Analysis**

Macro F1 = 0.22, the worst result overall. Recall=1.0 on every channel but precision collapses: T-3 has 8395 false positives, G-1 has 8348.

D-3 is the only channel with acceptable performance (F1=0.57) due to its large anomaly region. T-5 is the worst at F1=0.028 — the anomaly was found but buried under excessive false alarms.

## 6.8 Model 7: LSTM Autoencoder

Trained on normal data only; flags timesteps where reconstruction error (MSE) exceeds the 99th percentile. Uses a sequence window of 50 timesteps, which lets it catch contextual anomalies that point-based models miss. Column 0 only — using all 25 features dropped Macro F1 from 0.787 to 0.617.

In [ ]:
from LSTMAutoencoder import LSTM_AE_Detector

lstm = LSTM_AE_Detector(seq_len=50, hidden_dim=32, epochs=5, percentile=99, n_features=1)
lstm.fit(train_surv)
lstm_preds   = lstm.prediction(test_surv)
lstm_report  = evaluation.compare_methods_results(predictions_dict=lstm_preds)
evaluation.plot_hits_vs_misses(lstm_report, title='LSTM Autoencoder')
print(lstm_report[['Channel', 'Precision', 'Recall', 'F1_Score']].to_string(index=False))

**LSTM Autoencoder Analysis**

Macro F1 = 0.93, by far the best result. D-3, D-9, and P-2 hit F1=1.0; D-15 reaches 0.998. T-4, T-3, and G-7 are all above 0.95.

G-1 (F1=0.74) and T-5 (0.78) are the weaker channels — both have recall=1.0 but small anomaly windows (121 and 26 points respectively), so even a handful of false positives pulls precision down noticeably.

## STOMPY

In [ ]:
stmp = MstumpDetector(window_size=200)
stmp_error_prediction = stmp.get_batch_predictions(test_surv, threshold_percentile=95)
stmp_report = evaluation.compare_methods_results(stmp_error_prediction)
evaluation.plot_hits_vs_misses(stmp_report)
print(stmp_report)

## 6.9 Model 8: PCA preprocesing + LOF and GMM procesing

In [ ]:
# PCA preprocesing
bpca = BatchPCA(train_surv, test_surv)
bpca.fit_all(n_components=15)
PCA_test_errors = bpca.get_PCA_predictions(mode="test",threshold_percentile=95)
PCA_result = evaluation.compare_methods_results(predictions_dict=PCA_test_errors)
train_pca_features = bpca.transform_PCA(mode="train")
test_pca_features = bpca.transform_PCA(mode="test")

In [ ]:
# GMM
gmm = GMM(train_pca_features,test_pca_features)
gmm.fit_all(n_components=3, covariance_type='diag', n_init=5, max_iter=300, tol=1e-3, init_params='kmeans', warm_start=True)
GMM_errors_pred = gmm.get_batch_predictions(threshold_percentile=5)
GMM_results = evaluation.compare_methods_results(predictions_dict=GMM_errors_pred)
evaluation.plot_hits_vs_misses(GMM_results)
print(GMM_results)

In [ ]:
#LOF
lof = LOF(train_pca_features,test_pca_features)
lof.fit_all(n_neighbors=50, algorithm='auto', leaf_size=50, metric='minkowski', p=4, contamination='auto')
LOF_error_prediction = lof.get_batch_predictions(threshold_percentile=10)
LOF_results = evaluation.compare_methods_results(predictions_dict=LOF_error_prediction)
evaluation.plot_hits_vs_misses(LOF_results)
print(LOF_results)


# 7. Results Summary

## 7.1 Model Comparison

In [ ]:
def macro_f1(report_df):
    return round(report_df['F1_Score'].mean(), 4) if not report_df.empty else None

comparison = pd.DataFrame([
    {'Model': 'K-Means',           'Macro F1': macro_f1(km_report),   'Notes': 'threshold_percentile=95, K=5'},
    {'Model': 'DBSCAN',            'Macro F1': macro_f1(db_report),   'Notes': 'eps=2.0, min_samples=10'},
    {'Model': 'Isolation Forest',  'Macro F1': macro_f1(iso_report),  'Notes': 'contamination=0.10'},
    {'Model': 'One-Class SVM',     'Macro F1': macro_f1(svm_report),  'Notes': 'nu=0.05, kernel=rbf'},
    {'Model': 'LSTM Autoencoder',  'Macro F1': macro_f1(lstm_report), 'Notes': 'seq_len=50, hidden_dim=32, percentile=99'},
])

print(comparison.to_string(index=False))

# Bar chart
fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#d62728' if v < 0.5 else '#ff7f0e' if v < 0.7 else '#2ca02c'
          for v in comparison['Macro F1']]
bars = ax.barh(comparison['Model'], comparison['Macro F1'], color=colors, edgecolor='white')
ax.bar_label(bars, fmt='%.4f', padding=4, fontsize=10)
ax.set_xlim(0, 1.05)
ax.set_xlabel('Macro F1 Score (10-channel subset, point-adjust)')
ax.set_title('Model Comparison — Macro F1', fontsize=13, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

---
# 8. Online Deployment Discussion

### How to Use the Solution in Online (Real-Time) Mode

In a real NASA ground operations context, telemetry data streams in continuously from the spacecraft. Here is how the trained model would be integrated:

| Aspect | Approach |
|--------|----------|
| **Data ingestion** | Each incoming telemetry packet is appended to a rolling buffer of fixed length |
| **Preprocessing** | The buffer is transformed using the already-fitted `StandardScaler` (fitted on training data) |
| **Inference** | The preprocessed buffer is scored by the model (e.g., GMM log-likelihood or Isolation Forest score) |
| **Decision** | If the score exceeds the pre-calibrated threshold, the timestep is flagged as anomalous |
| **Alert** | An alert is forwarded to the spacecraft operations team for review against ISA reports |
| **Model refresh** | Periodically retrain on a recent window of confirmed-normal telemetry to account for mission phase changes |

**Model suitability for online use:**

| Model | Online-compatible? | Notes |
|-------|--------------------|-------|
| K-Means | Yes | Store centroids; compute distance per new sample |
| DBSCAN | No (without adaptation) | Requires access to full dataset at inference time |
| GMM | Yes | Evaluate log-likelihood per new sample — very fast |
| Isolation Forest | Yes | Score each sample independently after training |
| LOF | Yes (with `novelty=True`) | Requires storing training data for neighbour queries |

**Recommended online pipeline:**  
`Telemetry stream → Rolling buffer → StandardScaler → GMM or Isolation Forest → Threshold → Alert`

---
# 9. ML Project Checklist Report

### Frame the Problem
- [ ] Business objective defined (spacecraft anomaly detection — SMAP & MSL)
- [ ] Problem type identified (unsupervised anomaly detection on multivariate time series)
- [ ] Performance metrics selected (F1, Precision, Recall against labeled_anomalies.csv)

### Get the Data
- [ ] Dataset loaded from `archive/data/data/train/` and `archive/data/data/test/`
- [ ] Ground-truth labels loaded from `archive/labeled_anomalies.csv`
- [ ] Channel selection documented (which channels are analysed and why)

### Explore the Data
- [ ] Raw telemetry signals plotted with ground-truth anomaly windows overlaid
- [ ] Missing values checked
- [ ] Distributions inspected
- [ ] Correlations analysed
- [ ] Anomaly label breakdown analysed (point vs contextual, anomaly rate per channel)

### Prepare the Data
- [ ] Pre-defined train/test split confirmed (no manual splitting needed)
- [ ] Preprocessing pipeline fitted on training data only
- [ ] Feature engineering documented (rolling stats, lag features, or raw)
- [ ] Ground-truth label arrays constructed for the test set
- [ ] No data leakage from test to train confirmed

### Model Exploration
- [ ] K-Means implemented (elbow method for K, centroid distance as score)
- [ ] DBSCAN implemented (k-distance plot for eps)
- [ ] GMM implemented (BIC/AIC for n_components, log-likelihood as score)
- [ ] Isolation Forest implemented
- [ ] LOF implemented
- [ ] All models evaluated on same channel(s) using F1, Precision, Recall

### Fine-Tune
- [ ] Hyperparameters tuned for best models
- [ ] Threshold sensitivity analysed
- [ ] Ensemble / best model selected and justified
- [ ] Final anomalies visualised overlaid on raw telemetry

### Online Deployment
- [ ] Online pipeline described (buffer → scaler → model → threshold → alert)
- [ ] Model-by-model online suitability assessed
- [ ] Concept drift / model refresh strategy outlined

---
### Summary

**Best performing model:** [TBD]  
**Why it outperforms others:** [TBD — discuss point vs contextual anomalies, feature space geometry, etc.]  
**Recommended online model:** [TBD]  
**Key anomalies detected:** [TBD — describe what was found and on which channels]